In [2]:
import numpy as np
import pandas as pd
import datetime as dt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style(style='whitegrid')
sns.set_theme(rc={'figure.figsize':(11.7,8.27)})
pd.options.display.max_rows = 4000

In [3]:
appliances = pd.read_csv('../data/raw/w1_appliances.csv')
w2_appliances = pd.read_csv('../data/raw/w2_appliance_usage.csv')
w3_appliances = pd.read_csv('../data/raw/w3_appliance_usage.csv')

In [4]:
appliances['appliance_type'].value_counts().sort_index().to_frame()

,count
appliance_type,
Air fryer,190
Bluetooth Speakers,358
CCTV camera systems,827
Camera (that needs to be charged using electricity),53
Clothes dryer,128
Coffee maker,12
Computers,473
DVD / VCD,453
Dialog TV / Peo TV / Satellite TV box,1705


In [6]:
w2_appliances['appliance_type'].value_counts().sort_index().to_frame()

,count
appliance_type,
Air fryer,133
Bluetooth Speakers,254
"CCTV camera systems (note to int.: even if there are more than one camera in the system, please consider as 1 CCTV cam",530
Camera (that needs to be charged using electricity),34
Clothes dryer,91
Coffee maker,6
Computers,340
DVD / VCD,317
Dialog TV / Peo TV / Satellite TV box,1334


In [5]:
w3_appliances['appliance_type'].value_counts().sort_index().to_frame()

,count
appliance_type,
Air fryer,136
Bluetooth Speakers,264
CCTV camera systems,511
Camera (that needs to be charged using electricity),35
Clothes dryer,99
Coffee maker,6
Computers,365
DVD / VCD,316
Dialog TV / Peo TV / Satellite TV box,1300


In [7]:
import re

def clean_appliance_text(x):
    if pd.isna(x):
        return x
    
    x = x.lower().strip()
    x = re.sub(r"\s+", " ", x)          # remove extra spaces
    x = re.sub(r"[\(\)\:\,\-]", " ", x) # remove punctuation noise
    x = re.sub(r"\s+", " ", x).strip()
    
    return x

In [8]:
appliances["appliance_clean"] = appliances["appliance_type"].apply(clean_appliance_text)
w2_appliances["appliance_clean"] = w2_appliances["appliance_type"].apply(clean_appliance_text)
w3_appliances["appliance_clean"] = w3_appliances["appliance_type"].apply(clean_appliance_text)

In [9]:
heating = [
    "Air fryer",
    "Electric Iron including electric steam iron",
    "Electric Kettle",
    "Electric Oven",
    "Electric cook tops (induction cookers, Infra-red cookers, hot plates)",
    "Microwave",
    "Toaster / Sandwich toaster",
    "Electric grill",
    "Electric pressure cooker",
    "Electric water heater to heat water for drinking purposes",
    "Geyser / Hot water systems for bathrooms which operate from electricity",
    "Hair dryer",
    "Hair iron / hair curlers",
    "Electric coconut scraper",
    "Electric grinder",
    "Electric mixer / beater",
    "Electric food processor",
    "Electric blender",
    "Electric heater (to control room temperature)",
    "Coffee maker",
    "Waffle maker"
]

In [10]:
cooling = [
    "Refrigerator",
    "Separate Freezer",
    "Mini Bar",
    "Humidifier",
    "Electric Water filter / water dispenser",
    "Rice Cooker"
]

In [11]:
entertainment = [
    "TV",
    "Home theater system",
    "Sound systems (Subwoofer)/Stereo",
    "DVD / VCD",
    "Gaming console/PlayStation",
    "Computers",
    "Laptops",
    "Mobile phone - Smart phones",
    "Mobile phone - Feature phones",
    "Mobile phone - Basic phones",
    "Bluetooth Speakers",
    "Radio",
    "Tab",
    "Camera (that needs to be charged using electricity)",
    "Electric musical Instruments (ex.: electric organ, electric guitar etc.)",
    "Dialog TV / Peo TV / Satellite TV box",
    "TV antenna"
]

In [12]:
cleaning = [
    "Washing Machine",
    "Dish washer",
    "Electric Vacuum Cleaner",
    "Electric Sewing machine",
    "Electric Water pump",
    "Electric Bell",
    "Electric floor polisher",
    "Electric Lawn mower",
    "Electric water gun (used to wash cars etc.)",
    "Clothes Dryer"
]

In [13]:
communication = [
    "Routers",
    "Printer",
    "Scanner",
    "Photo Copiers",
    "Fax machines",
    "Fixed phones",
    "Power banks",
]

In [14]:
security = [
    "CCTV camera systems",
    "Electric Alarm System",
    "Other Electric Security Systems",
    "Electrical exhaust fan fitted above the oven or the hot plate",
    "Electric Fountain / decorative waterfall",
]

In [15]:
mobility = [
    "Electric vehicles (four wheelers) - cars,  vans, SUVs)",
    "Electric vehicles (three wheelers)",
    "Electric vehicles (two wheelers) - Electric bicycles, scooters",
    "Emergency Light / re-chargeable torches",
]

In [16]:
other = [
    "Other1",
    "Other 2",
    "Roller door",
    "Oxygen filter for fish tank",
    "Hot tub",
    "Toys with re-chargeable batteries"
]

In [17]:
appliance_mapping = {}

for item in heating:
    appliance_mapping[item] = "Heating_Cooking"

for item in cooling:
    appliance_mapping[item] = "Cooling_Refrigeration"

for item in entertainment:
    appliance_mapping[item] = "Entertainment_ICT"

for item in cleaning:
    appliance_mapping[item] = "Cleaning_Household"

for item in communication:
    appliance_mapping[item] = "Communication_IT"

for item in security:
    appliance_mapping[item] = "Security_Infrastructure"

for item in mobility:
    appliance_mapping[item] = "Mobility_Energy"

for item in other:
    appliance_mapping[item] = "Other"

In [18]:
appliances.head()

,household_ID,appliance_ID,appliance_type,no_of_hours_used_during_last_week,appliance_clean
0,ID0001,O1_1,Refrigerator,84.0,refrigerator
1,ID0001,O12_1,Rice cooker,3.0,rice cooker
2,ID0001,O26_1,Electric Iron including electric steam iron,1.0,electric iron including electric steam iron
3,ID0001,O31_1,TV,0.0,tv
4,ID0001,O45_1,Mobile phone - Smart phones,5.0,mobile phone smart phones


In [19]:
appliance_mapping_clean = {
    clean_appliance_text(k): v for k, v in appliance_mapping.items()
}

In [23]:
for dataset in [appliances, w2_appliances, w3_appliances]:
    dataset["appliance_clean"] = dataset["appliance_type"].apply(clean_appliance_text)
    dataset["appliance_category"] = dataset["appliance_clean"].map(appliance_mapping_clean)
    dataset["appliance_category"] = dataset["appliance_category"].fillna("Other")

In [24]:
unmapped = appliances[appliances["appliance_category"] == "Other"]["appliance_clean"].unique()
print(unmapped)

['oxygen filter for fish tank' 'toys with re chargeable batteries'
 'roller door' 'hot tub' 'electric shavers' 'other 2'
 'electric exercise machines' 'other1']


In [25]:
unmapped = w2_appliances[appliances["appliance_category"] == "Other"]["appliance_clean"].unique()
print(unmapped)

['refrigerator'
 'electrical exhaust fan fitted above the oven or the hot plate'
 'electric blender' 'electric grinder'
 'electric iron including electric steam iron' 'rice cooker'
 'electric kettle' 'laptops' 'tv' 'tv antenna' 'power banks'
 'mobile phone – smart phones' 'routers' 'mobile phone – feature phones'
 'printer' 'mobile phone – basic phones' 'electric oven'
 'electric musical instruments ex. electric organ electric guitar etc.'
 'dialog tv / peo tv / satellite tv box' 'other1' 'other 2'
 'home theater system' 'washing machine' 'fixed phones'
 'electric water pump' 'toaster / sandwich toaster' 'dish washer' 'radio'
 'dvd / vcd' 'emergency light / re chargeable torches' 'microwave'
 'electric vacuum cleaner' 'electric sewing machine'
 'oxygen filter for fish tank'
 'geyser / hot water systems for bathrooms which operate from electricity'
 'electric water heater to heat water for drinking purposes'
 'electric coconut scraper' 'electric shavers' 'electric bell'
 'electric cook 

In [26]:
unmapped = w3_appliances[appliances["appliance_category"] == "Other"]["appliance_clean"].unique()
print(unmapped)

['mobile phone – smart phones'
 'electric iron including electric steam iron' 'tv'
 'dialog tv / peo tv / satellite tv box' 'mobile phone – basic phones'
 'refrigerator' 'laptops' 'routers' 'tv antenna' 'electric blender'
 'washing machine' 'radio' 'cctv camera systems'
 'toaster / sandwich toaster' 'electric oven' 'fixed phones'
 'geyser / hot water systems for bathrooms which operate from electricity'
 'bluetooth speakers' 'electric sewing machine' 'rice cooker'
 'mobile phone – feature phones' 'toys with re chargeable batteries'
 'electric water heater to heat water for drinking purposes'
 'electric bell' 'electric kettle'
 'electric musical instruments ex. electric organ electric guitar etc.'
 'electric mixer / beater' 'air fryer' 'electric shavers'
 'electric water pump' 'microwave' 'electric vacuum cleaner' 'computers'
 'hair dryer' 'other1' 'scanner' 'dvd / vcd'
 'emergency light / re chargeable torches' 'electric coconut scraper'
 'power banks' 'sound systems subwoofer /stereo'

In [27]:
appliances.head()

,household_ID,appliance_ID,appliance_type,no_of_hours_used_during_last_week,appliance_clean,appliance_category
0,ID0001,O1_1,Refrigerator,84.0,refrigerator,Cooling_Refrigeration
1,ID0001,O12_1,Rice cooker,3.0,rice cooker,Cooling_Refrigeration
2,ID0001,O26_1,Electric Iron including electric steam iron,1.0,electric iron including electric steam iron,Heating_Cooking
3,ID0001,O31_1,TV,0.0,tv,Entertainment_ICT
4,ID0001,O45_1,Mobile phone - Smart phones,5.0,mobile phone smart phones,Entertainment_ICT


In [28]:
print(sorted(w2_appliances["appliance_clean"].unique()))

['air fryer', 'bluetooth speakers', 'camera that needs to be charged using electricity', 'cctv camera systems note to int. even if there are more than one camera in the system please consider as 1 cctv cam', 'clothes dryer', 'coffee maker', 'computers', 'dialog tv / peo tv / satellite tv box', 'dish washer', 'dvd / vcd', 'electric alarm system', 'electric bell', 'electric blender', 'electric coconut scraper', 'electric cook tops induction cookers infra red cookers hot plates', 'electric exercise machines', 'electric floor polisher', 'electric food processor', 'electric fountain / decorative waterfall', 'electric grill', 'electric grinder', 'electric heater to control room temperature', 'electric iron including electric steam iron', 'electric kettle', 'electric lawn mower', 'electric mixer / beater', 'electric musical instruments ex. electric organ electric guitar etc.', 'electric oven', 'electric pressure cooker', 'electric sewing machine', 'electric shavers', 'electric vacuum cleaner'

In [29]:
print(sorted(w3_appliances["appliance_clean"].unique()))

['air fryer', 'bluetooth speakers', 'camera that needs to be charged using electricity', 'cctv camera systems', 'clothes dryer', 'coffee maker', 'computers', 'dialog tv / peo tv / satellite tv box', 'dish washer', 'dvd / vcd', 'electric alarm system', 'electric bell', 'electric blender', 'electric coconut scraper', 'electric cook tops induction cookers infra red cookers hot plates', 'electric exercise machines', 'electric floor polisher', 'electric food processor', 'electric fountain / decorative waterfall', 'electric grill', 'electric grinder', 'electric heater to control room temperature', 'electric iron including electric steam iron', 'electric kettle', 'electric lawn mower', 'electric mixer / beater', 'electric musical instruments ex. electric organ electric guitar etc.', 'electric oven', 'electric pressure cooker', 'electric sewing machine', 'electric shavers', 'electric vacuum cleaner', 'electric vehicles four wheelers cars vans suvs', 'electric vehicles two wheelers electric bic

In [30]:
def map_appliance_fuzzy(x):
    if pd.isna(x):
        return "Other"
    
    x = x.lower()

    # Heating
    if any(k in x for k in ["iron", "kettle", "oven", "cook", "microwave", "heater", "toaster", "grill", "electric grinder", "electric blender", "electric mixer / beater", "air fryer", "hot water systems", "coffee maker", "coconut scraper", "waffle maker", "food processor"]):
        return "Heating_Cooking"

    # Cooling
    if any(k in x for k in ["fridge", "refrigerator", "freezer", "mini bar", "humidifier", "water filter"]):
        return "Cooling_Refrigeration"

    # Entertainment
    if any(k in x for k in ["tv", "radio", "laptop", "computer", "phone", "speaker", "dvd", "console","sound systems subwoofer /stereo", "camera", "instruments", "exercise machines"]):
        return "Entertainment_ICT"

    # Cleaning
    if any(k in x for k in ["washing", "vacuum", "pump", "sewing", "dryer", "electric water gun used to wash cars etc.","shavers", "floor polisher", "dish washer", "lawn mower"]):
        return "Cleaning_Household"

    # Communication
    if any(k in x for k in ["router", "printer", "scanner", "fax", "power bank","photo copiers"]):
        return "Communication_IT"

    # Security
    if any(k in x for k in ["cctv", "alarm", "security","electric bell", "emergency light / re chargeable torches"]):
        return "Security_Infrastructure"

    # Mobility
    if any(k in x for k in ["vehicle", "battery", "scooter"]):
        return "Mobility_Energy"

    return "Other"

In [31]:
w2_appliances["category"] = w2_appliances["appliance_clean"].apply(map_appliance_fuzzy)
w3_appliances["category"] = w3_appliances["appliance_clean"].apply(map_appliance_fuzzy)

In [32]:
unmapped_w2 = w3_appliances[w3_appliances["category"] == "Other"]["appliance_clean"].unique()
print(unmapped_w2)

['oxygen filter for fish tank' 'hot tub' 'other1' 'other 2' 'roller door'
 'other 3' 'tab' 'toys with re chargeable batteries'
 'electric fountain / decorative waterfall']


In [33]:
w1_features = appliances.groupby(["household_ID", "appliance_category"]) \
                        .size() \
                        .unstack(fill_value=0)

In [34]:
w1_features["wave"] = 1

In [35]:
w2_features = w2_appliances.groupby(["household_ID", "appliance_category"]) \
                        .size() \
                        .unstack(fill_value=0)

In [36]:
w2_features["wave"] = 2

In [37]:
w3_features = w3_appliances.groupby(["household_ID", "appliance_category"]) \
                        .size() \
                        .unstack(fill_value=0)


In [38]:
w3_features["wave"] = 3

In [39]:
all_cols = set(w1_features.columns) | set(w2_features.columns) | set(w3_features.columns)

w1_features = w1_features.reindex(columns=all_cols, fill_value=0)
w2_features = w2_features.reindex(columns=all_cols, fill_value=0)
w3_features = w3_features.reindex(columns=all_cols, fill_value=0)

In [40]:
appliance_panel = pd.concat([w1_features, w2_features, w3_features])

In [41]:
appliance_panel.head()

appliance_category,Heating_Cooking,wave,Communication_IT,Mobility_Energy,Cooling_Refrigeration,Cleaning_Household,Security_Infrastructure,Entertainment_ICT,Other
household_ID,,,,,,,,,
ID0001,1,1,0,0,2,0,0,5,0
ID0002,0,1,0,0,2,0,0,2,0
ID0003,6,1,3,1,2,2,0,5,0
ID0004,6,1,0,0,2,2,0,6,0
ID0005,3,1,1,0,3,1,0,3,0


In [42]:
appliance_panel.tail()

appliance_category,Heating_Cooking,wave,Communication_IT,Mobility_Energy,Cooling_Refrigeration,Cleaning_Household,Security_Infrastructure,Entertainment_ICT,Other
household_ID,,,,,,,,,
ID4060,1,3,0,0,2,0,0,2,1
ID4061,0,3,0,0,0,0,0,1,1
ID4062,1,3,1,0,1,1,0,2,1
ID4063,3,3,1,0,2,1,0,3,7
ID5212,1,3,0,0,0,0,0,3,4


In [43]:
survey_dates = pd.read_csv('../data/raw/survey_dates.csv')

In [44]:
survey_dates_long = survey_dates.melt(
    id_vars='household_ID',
    value_vars=['wave1', 'wave2', 'wave3'],
    var_name='wave',
    value_name='survey_date'
)

# Clean wave column (wave1 → 1)
survey_dates_long['wave'] = survey_dates_long['wave'].str.extract('(\d)').astype(float)

survey_dates_long['survey_date'] = pd.to_datetime(
    survey_dates_long['survey_date'], errors='coerce'
)

In [45]:
appliance_panel = appliance_panel.merge(
    survey_dates_long,
    on=['household_ID', 'wave'],
    how='left'
)

In [46]:
print(appliance_panel.columns)

Index(['household_ID', 'Heating_Cooking', 'wave', 'Communication_IT',
       'Mobility_Energy', 'Cooling_Refrigeration', 'Cleaning_Household',
       'Security_Infrastructure', 'Entertainment_ICT', 'Other', 'survey_date'],
      dtype='object')


In [47]:
appliance_panel.head()

,household_ID,Heating_Cooking,wave,Communication_IT,Mobility_Energy,Cooling_Refrigeration,Cleaning_Household,Security_Infrastructure,Entertainment_ICT,Other,survey_date
0,ID0001,1,1,0,0,2,0,0,5,0,2023-12-09
1,ID0002,0,1,0,0,2,0,0,2,0,2023-12-09
2,ID0003,6,1,3,1,2,2,0,5,0,2023-12-09
3,ID0004,6,1,0,0,2,2,0,6,0,2023-12-09
4,ID0005,3,1,1,0,3,1,0,3,0,2023-12-09


In [48]:
appliance_panel = appliance_panel.dropna(subset=['survey_date'])

In [49]:
appliance_panel['total_appliances'] = appliance_panel[
    [
        'Security_Infrastructure',
        'Entertainment_ICT',
        'Heating_Cooking',
        'Communication_IT',
        'Cooling_Refrigeration',
        'Mobility_Energy',
        'Cleaning_Household'
    ]
].sum(axis=1)

In [50]:
appliance_panel['high_energy_appliances'] = (
    appliance_panel['Heating_Cooking'] +
    appliance_panel['Cooling_Refrigeration'] +
    appliance_panel['Cleaning_Household']
)

In [51]:
appliance_panel['cooling_ratio'] = (
    appliance_panel['Cooling_Refrigeration'] / appliance_panel['total_appliances']
)

In [52]:
appliance_panel['digital_lifestyle'] = (
    appliance_panel['Entertainment_ICT'] +
    appliance_panel['Communication_IT']
)

In [53]:
appliance_panel['appliance_diversity'] = (
    (appliance_panel[
        [
            'Security_Infrastructure',
            'Entertainment_ICT',
            'Heating_Cooking',
            'Communication_IT',
            'Cooling_Refrigeration',
            'Mobility_Energy',
            'Cleaning_Household'
        ]
    ] > 0).sum(axis=1)
)

In [54]:
appliance_panel = appliance_panel.sort_values(['household_ID', 'survey_date'])

In [56]:
appliance_panel.head(15)

,household_ID,Heating_Cooking,wave,Communication_IT,Mobility_Energy,Cooling_Refrigeration,Cleaning_Household,Security_Infrastructure,Entertainment_ICT,Other,survey_date,total_appliances,high_energy_appliances,cooling_ratio,digital_lifestyle,appliance_diversity
0,ID0001,1,1,0,0,2,0,0,5,0,2023-12-09,8,3,0.250000,5,3
4055,ID0001,1,2,0,0,2,0,0,1,4,2024-09-30,4,3,0.500000,1,3
7447,ID0001,2,3,0,0,2,0,0,1,4,2024-11-28,5,4,0.400000,1,3
1,ID0002,0,1,0,0,2,0,0,2,0,2023-12-09,4,2,0.500000,2,2
4056,ID0002,0,2,0,0,2,0,0,1,2,2024-10-02,3,2,0.666667,1,2
7448,ID0002,0,3,0,0,4,0,0,1,2,2024-12-11,5,4,0.800000,1,2
2,ID0003,6,1,3,1,2,2,0,5,0,2023-12-09,19,10,0.105263,8,6
4057,ID0003,6,2,3,1,2,2,0,2,1,2024-10-06,16,10,0.125000,5,6
7449,ID0003,6,3,3,1,2,2,0,4,1,2025-01-12,18,10,0.111111,7,6
3,ID0004,6,1,0,0,2,2,0,6,0,2023-12-09,16,10,0.125000,6,4


In [57]:
appliance_panel.to_csv("../data/processed1/appliances.csv", index=False)